# 08 — Traditional, deep learning and stacking comparison


## 1. Match the development comparison



In [1]:
from pathlib import Path
import json
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import GridSearchCV, ParameterGrid, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from threadpoolctl import threadpool_limits

root = Path.cwd().resolve()
if root.name == 'notebooks':
    root = root.parent
cleaned = root / 'data' / 'processed' / 'loan_default_cleaned.csv'
if not cleaned.is_file():
    raise FileNotFoundError('Open Notebook 08 from the project root or its notebooks folder.')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.data_preparation import TrainingFoldPreprocessor, split_loan_data
from src.model_tuning import SEED, BalancedTrainingPipeline
from src.model_validation import top_fraction_predictions

tables = root / 'outputs' / 'tables' / 'model_types'
tables.mkdir(parents=True, exist_ok=True)
X_dev, X_reserved, y_dev, _ = split_loan_data(pd.read_csv(cleaned))
assert X_dev.index.intersection(X_reserved.index).empty
assert len(X_dev) + len(X_reserved) == len(pd.read_csv(cleaned))
outer_folds = list(StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED).split(X_dev, y_dev))
coverage = np.concatenate([validation for _, validation in outer_folds])
np.testing.assert_array_equal(np.sort(coverage), np.arange(len(X_dev)))

reference = pd.read_csv(root / 'outputs' / 'tables' / 'part_g' / 'fold_scores.csv')
reference = reference.loc[reference['model'].isin(['XGBoost', 'Stacking'])].copy()
assert len(reference) == 6 and not reference.duplicated(['model', 'fold']).any()
print(f'Development rows used: {len(X_dev):,} | Reserved rows excluded: {len(X_reserved):,}')

Development rows used: 45,547 | Reserved rows excluded: 11,387


## 2. Grid-search the deep network


In [2]:
grid = {
    'model__hidden_layer_sizes': [(32, 16, 8), (64, 32, 16)],
    'model__alpha': [0.0001, 0.01],
    'balanced': [False, True],
}
display(pd.DataFrame(ParameterGrid(grid)))

pipeline = BalancedTrainingPipeline([
    ('prepare', TrainingFoldPreprocessor(log_income=True)),
    ('model', MLPClassifier(
        activation='relu', solver='adam', learning_rate_init=0.002, batch_size=512,
        max_iter=60, tol=0.0005, n_iter_no_change=8, random_state=SEED,
    )),
])

candidate_rows, deep_rows = [], []
for fold, (fit, validation) in enumerate(outer_folds, start=1):
    assert not np.intersect1d(fit, validation).size
    X_fit, X_val = X_dev.iloc[fit], X_dev.iloc[validation]
    y_fit, y_val = y_dev.iloc[fit], y_dev.iloc[validation]
    inner = StratifiedKFold(n_splits=2, shuffle=True, random_state=SEED + fold)
    search = GridSearchCV(pipeline, grid, scoring='average_precision', cv=inner,
                          refit=True, error_score='raise', n_jobs=1)
    with threadpool_limits(limits=1, user_api='blas'), warnings.catch_warnings(record=True) as recorded:
        warnings.simplefilter('always', ConvergenceWarning)
        search.fit(X_fit, y_fit)
        p = search.predict_proba(X_val)[:, 1]
    cap_warnings = sum(issubclass(item.category, ConvergenceWarning) for item in recorded)
    for item in recorded:
        if not issubclass(item.category, ConvergenceWarning):
            warnings.warn(item.message, item.category)
    assert np.array_equal(search.classes_, [0, 1])
    assert np.isfinite(p).all() and np.all((p >= 0) & (p <= 1))
    model = search.best_estimator_.named_steps['model']
    assert len(model.coefs_) == 4  # three hidden layers plus output
    results = search.cv_results_
    candidate_rows.extend({
        'fold': fold, 'parameters': json.dumps(params, sort_keys=True),
        'inner_split_1_ap': results['split0_test_score'][i],
        'inner_split_2_ap': results['split1_test_score'][i],
        'mean_inner_ap': results['mean_test_score'][i],
        'rank': results['rank_test_score'][i], 'selected': i == search.best_index_,
    } for i, params in enumerate(results['params']))

    queue = top_fraction_predictions(p).astype(bool)
    caught = int(y_val.to_numpy()[queue].sum())
    deep_rows.append({
        'fold': fold, 'model': 'Deep MLP', 'ap': average_precision_score(y_val, p),
        'roc_auc': roc_auc_score(y_val, p), 'brier': brier_score_loss(y_val, p),
        'reviewed': int(queue.sum()), 'defaults_in_fold': int(y_val.sum()), 'defaults_caught': caught,
        'top10_precision': caught / queue.sum(), 'top10_recall': caught / y_val.sum(),
        'hidden_layers': str(model.hidden_layer_sizes), 'alpha': model.alpha,
        'balanced': search.best_params_['balanced'], 'epochs': model.n_iter_, 'cap_warnings_in_search': cap_warnings,
    })
    print(f'Fold {fold}: {model.hidden_layer_sizes}, alpha={model.alpha}, epochs={model.n_iter_}, '
          f'iteration-cap warnings across search fits: {cap_warnings}', flush=True)

candidates = pd.DataFrame(candidate_rows)
deep = pd.DataFrame(deep_rows)
assert len(candidates) == 3 * len(list(ParameterGrid(grid))) and candidates.groupby('fold')['selected'].sum().eq(1).all()
candidates.to_csv(tables / 'deep_grid_results.csv', index=False)
deep.to_csv(tables / 'deep_model_folds.csv', index=False)
display(deep[['fold', 'hidden_layers', 'alpha', 'epochs', 'cap_warnings_in_search', 'ap']].round(4))

,balanced,model__alpha,model__hidden_layer_sizes
0,False,0.0001,"(32, 16, 8)"
1,False,0.0001,"(64, 32, 16)"
2,False,0.0100,"(32, 16, 8)"
3,False,0.0100,"(64, 32, 16)"
4,True,0.0001,"(32, 16, 8)"
5,True,0.0001,"(64, 32, 16)"
6,True,0.0100,"(32, 16, 8)"
7,True,0.0100,"(64, 32, 16)"


Fold 1: (32, 16, 8), alpha=0.0001, epochs=60, iteration-cap warnings across search fits: 17


Fold 2: (32, 16, 8), alpha=0.01, epochs=60, iteration-cap warnings across search fits: 17


Fold 3: (32, 16, 8), alpha=0.0001, epochs=60, iteration-cap warnings across search fits: 17


,fold,hidden_layers,alpha,epochs,cap_warnings_in_search,ap
0,1,"(32, 16, 8)",0.0001,60,17,0.5971
1,2,"(32, 16, 8)",0.0100,60,17,0.6130
2,3,"(32, 16, 8)",0.0001,60,17,0.5718


## 3. Compare the three types


In [3]:
matched = pd.concat([reference, deep[reference.columns]], ignore_index=True)
assert matched.groupby('fold')['defaults_in_fold'].nunique().eq(1).all()
assert matched.groupby('fold')['reviewed'].nunique().eq(1).all()
types = {'XGBoost': 'Traditional', 'Deep MLP': 'Deep learning', 'Stacking': 'Stacking'}
matched.insert(1, 'model_type', matched['model'].map(types))
matched.to_csv(tables / 'matched_fold_scores.csv', index=False)
summary = matched.groupby(['model_type', 'model']).agg(
    mean_ap=('ap', 'mean'), fold_ap_sd=('ap', 'std'), mean_roc_auc=('roc_auc', 'mean'),
    mean_brier=('brier', 'mean'), mean_top10_recall=('top10_recall', 'mean'),
    defaults_caught=('defaults_caught', 'sum'), total_reviews=('reviewed', 'sum'),
).sort_values('mean_ap', ascending=False)
summary.to_csv(tables / 'model_type_comparison.csv')
display(summary.round(4))

fold_ap = matched.pivot(index='fold', columns='model', values='ap')
display((100 * fold_ap[['Deep MLP', 'Stacking']].sub(fold_ap['XGBoost'], axis=0)).round(3)
        .rename_axis('Fold | AP change versus XGBoost (percentage points)'))

,,mean_ap,fold_ap_sd,mean_roc_auc,mean_brier,mean_top10_recall,defaults_caught,total_reviews
model_type,model,,,,,,,
Traditional,XGBoost,0.6159,0.0155,0.8136,0.1001,0.4156,3269,4557
Stacking,Stacking,0.6156,0.0156,0.8155,0.1007,0.4137,3254,4557
Deep learning,Deep MLP,0.5940,0.0208,0.8005,0.1051,0.3983,3133,4557


model,Deep MLP,Stacking
Fold | AP change versus XGBoost (percentage points),,
1,-2.600,0.078
2,-1.353,-0.123
3,-2.629,-0.052
